In [10]:
from agents import *
from routing import distrito_tec,get_closest_place_node_id
import numpy as np
import random
sub_graph,routable_restaurants,residential_zones = distrito_tec()


In [11]:
routable_restaurants.head(5)

,element,id,geometry,amenity,brand,brand:wikidata,brand:wikipedia,cuisine,name,official_name,...,air_conditioning,website:menu,contact:instagram,phone,drive_through,building,fast_food,building:levels,height,nn
0,node,890657171,POINT (-100.29383 25.65435),cafe,Starbucks,Q37158,en:Starbucks,coffee_shop,Starbucks,Starbucks Coffee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
1,node,890657529,POINT (-100.29377 25.65441),restaurant,NaN,NaN,NaN,NaN,Costeñito,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
2,node,890657855,POINT (-100.29355 25.65472),restaurant,NaN,NaN,NaN,NaN,Wings Army,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
3,node,890666591,POINT (-100.28436 25.64771),restaurant,NaN,NaN,NaN,NaN,Manhattan,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.049531e+09
4,node,890667172,POINT (-100.28432 25.64783),bar,NaN,NaN,NaN,NaN,La Rambla,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.049531e+09


In [12]:
routable_restaurants.shape

(50, 40)

In [13]:
residential_zones.head(5)

,element,id,geometry,landuse,name,place,addr:city,addr:street,building:levels,residential,operator,type
0,relation,9463437,"POLYGON ((-100.29348 25.66187, -100.2936 25.66...",residential,La Florida,NaN,NaN,NaN,NaN,NaN,NaN,multipolygon
1,way,163034600,"POLYGON ((-100.27483 25.62377, -100.27426 25.6...",residential,Contry,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,way,163034602,"POLYGON ((-100.27739 25.64568, -100.27511 25.6...",residential,Contry Tesoro,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,way,163034603,"POLYGON ((-100.28282 25.64297, -100.28238 25.6...",residential,Contry Lux,neighbourhood,NaN,NaN,NaN,NaN,NaN,NaN
4,way,163034605,"POLYGON ((-100.28052 25.64492, -100.27925 25.6...",residential,Las Musas,neighbourhood,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
routable_restaurants.iloc[[0]]

,element,id,geometry,amenity,brand,brand:wikidata,brand:wikipedia,cuisine,name,official_name,...,air_conditioning,website:menu,contact:instagram,phone,drive_through,building,fast_food,building:levels,height,nn
0,node,890657171,POINT (-100.29383 25.65435),cafe,Starbucks,Q37158,en:Starbucks,coffee_shop,Starbucks,Starbucks Coffee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09


In [ ]:
# =========================================================
# SCENARIO SETUP (users, drivers, restaurants, demand rate)
# =========================================================

# ---- PARAMETERS ----
N_USERS = 10000
N_DRIVERS = 200
ORDER_RATE = 15        # orders per minute

# ---------------------------------------------------------
# 1. Initialize simulation
# ---------------------------------------------------------

sim = Simulation(step_size=10, graph=sub_graph)

# ---------------------------------------------------------
# 2. Add restaurants (same as before)
# ---------------------------------------------------------

for i in range(len(routable_restaurants)):
    
    res_node = get_closest_place_node_id(
        routable_restaurants.iloc[[i]],
        sub_graph
    )

    res = Restaurant(
        restaurant_id=i,
        location=res_node,
        rating=5,
        capacity=100,
        avg_prep_time=300,
        service_radius=50000
    )

    sim.add_restaurant(res)

print(f"{len(sim.restaurants)} restaurants loaded")


# ---------------------------------------------------------
# 3. Generate residential nodes for users/drivers
# ---------------------------------------------------------

sampled_zones = residential_zones.sample(N_USERS, replace=True)

residential_nodes = [
    get_closest_place_node_id(sampled_zones.iloc[[i]], sub_graph)
    for i in range(N_USERS)
]


# ---------------------------------------------------------
# 4. Create users
# ---------------------------------------------------------

for i in range(N_USERS):

    user = User(
        user_id=i,
        location=residential_nodes[i]
    )

    sim.add_user(user)

print(f"{len(sim.users)} users created")


# ---------------------------------------------------------
# 5. Create drivers
# ---------------------------------------------------------

for i in range(N_DRIVERS):

    start_node = random.choice(residential_nodes)

    driver = Driver(
        driver_id=i,
        location_node=start_node,
        speed=None      # lets your normal distribution pick speed
    )

    sim.add_driver(driver)

print(f"{len(sim.drivers)} drivers created")


# ---------------------------------------------------------
# 6. Simulation now starts EMPTY
# Orders will appear dynamically each tick via:
#
# generate_orders(sim, ORDER_RATE)
#
# which should be called inside your animation update().
# ---------------------------------------------------------

print("Simulation initialized.")
print("Orders will now be generated dynamically by the Poisson process.")

50 restaurants loaded
10000 users created
200 drivers created
Simulation initialized.
Orders will now be generated dynamically by the Poisson process.


In [16]:
SHOW_TRAILS = False

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.cm as cm
import numpy as np
import osmnx as ox
from IPython.display import HTML
import pandas as pd # Necesario para procesar el historial

# --- 1. CONFIGURACIÓN Y LÍMITES DE MEMORIA ---
# to_jshtml() compila un video gigante en memoria. Subimos el límite preventivamente.

# --- 2. SETUP DEL MAPA Y ESTILOS ---
# Usamos un fondo negro y calles tenues para que los conductores resalten.
fig, ax = ox.plot_graph(sub_graph, bgcolor='k', edge_color='#333333', 
                        node_size=0, edge_linewidth=0.5, show=False, close=False)


# ---------------------------------------------------------
# Draw restaurant markers (static)
# ---------------------------------------------------------

restaurant_lons = []
restaurant_lats = []

for res in sim.restaurants.values():
    
    node_data = sub_graph.nodes[res.location]

    restaurant_lons.append(node_data['x'])
    restaurant_lats.append(node_data['y'])

ax.scatter(
    restaurant_lons,
    restaurant_lats,
    s=120,
    c='red',
    marker='^',
    edgecolors='white',
    linewidth=1.2,
    zorder=12,
    label='Restaurants'
)


# Identificamos a los conductores que tienen órdenes asignadas (están activos)
drivers_to_watch = list(sim.drivers.values())
if not drivers_to_watch:
    raise ValueError("No hay conductores con órdenes asignadas para animar. Revisa la lógica anterior.")

colors = cm.rainbow(np.linspace(0, 1, len(drivers_to_watch)))

# Elementos visuales: Un punto y una línea por cada conductor activo.
dots = ax.scatter([], [], c=[], s=60, zorder=10, edgecolors='white', linewidth=0.8)

if SHOW_TRAILS:
    lines = [ax.plot([], [], color=colors[i], lw=2.5, alpha=0.7, zorder=9)[0] for i in range(len(drivers_to_watch))]
else: 
    lines = []
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, color='white', fontsize=14, fontweight='bold')

# --- 3. HISTORIAL DE POSICIONES (Para el Trail) ---
# Almacenamos el rastro como una lista de listas de tuplas [(lon, lat), ...].
trail_history = [[] for _ in drivers_to_watch] if SHOW_TRAILS else None

# --- 4. LÓGICA DE ANIMACIÓN (Evolución de Ticks) ---
# Usamos blit=True para re-dibujar solo las partes que cambian, acelerando el proceso.
def update(frame):
    # AVANCE FÍSICO: La simulación avanza 10 segundos en cada frame.
    # Generate stochastic demand
    generate_orders(sim, rate_per_minute=40)

    # Advance simulation
    generate_orders(sim, ORDER_RATE)
    sim.run_tick()

    current_lats = []
    current_lons = []
    
    # Procesar la posición de cada conductor
    for i, driver in enumerate(drivers_to_watch):
        # 4a. Obtener posición actual calculada en agents.py
        if driver.coords != (0.0, 0.0):
            lon, lat = driver.coords[1], driver.coords[0]
        else:
            # Fallback al nodo inicial si aún no se ha movido
            node_data = sub_graph.nodes[driver.location]
            lon, lat = node_data['x'], node_data['y']
            
        current_lons.append(lon)
        current_lats.append(lat)
        
        # 4b. Actualizar el rastro histórico
        if SHOW_TRAILS:
            trail_history[i].append((lon, lat))
        
            # 4c. Actualizar la geometría de la línea (trail)
            # Convertimos la lista de tuplas en dos secuencias (X, Y) para set_data
            h_x, h_y = zip(*trail_history[i])
            lines[i].set_data(h_x, h_y)

    # Actualizar la posición de los puntos principales
    dots.set_offsets(np.c_[current_lons, current_lats])
    dots.set_color(colors)

    preparing = 0
    ready = 0
    picked = 0
    delivered = 0

    for o in sim.orders.values():
        if o.status == "PREPARING":
            preparing += 1
        elif o.status == "READY":
            ready += 1
        elif o.status == "PICKED_UP":
            picked += 1
        elif o.status == "DELIVERED":
            delivered += 1

    time_text.set_text(
        f'Time: {sim.current_time}s | P: {preparing} R: {ready} P: {picked} D: {delivered}'
    )

    
    # Devolvemos todos los objetos actualizados para que blit funcione
    return ([dots, time_text] + lines) if SHOW_TRAILS else [dots, time_text]

# --- 5. GENERACIÓN Y RENDER HTML ---
# frames=100 con step_size=10 animará 1000 segundos de simulación.
# interval=50ms (20fps) da una sensación de velocidad realista.
ani = FuncAnimation(fig, update, frames=300, interval=500, blit=True, repeat=False)

# Evita que Jupyter muestre una imagen estática extra antes del video
plt.close()

# Genera el reproductor de video HTML5 sólido
ani.save("sim.mp4")